# Model 1: Keypoints to Gloss (Computer Vision)

This notebook trains a 1D-CNN + BiLSTM model to translate MediaPipe spatial/velocity features into ASL Gloss using CTC Loss.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import re

# 1. Configuration
FEATURE_DIR = "how2sign_features/test"
GLOSS_CSV = "how2sign_realigned_test_glosses.csv"
BATCH_SIZE = 16
HIDDEN_DIM = 256
EPOCHS = 50
LR = 1e-3
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print(f"Using device: {DEVICE}")

In [ ]:
# 2. Data Loading & Vocabulary Building
df = pd.read_csv(GLOSS_CSV, sep='\t')
# Filter out rows with missing generated glosses
df = df.dropna(subset=['GENERATED_GLOSS'])

# Build Vocabulary
all_words = []
for gloss in df['GENERATED_GLOSS']:
    words = str(gloss).split()
    all_words.extend(words)

vocab = Counter(all_words)
# We reserve 0 for CTC Blank, 1 for <UNK>
word_to_idx = {"<BLANK>": 0, "<UNK>": 1}
for word, count in vocab.most_common():
    word_to_idx[word] = len(word_to_idx)
idx_to_word = {i: w for w, i in word_to_idx.items()}
num_classes = len(word_to_idx)

print(f"Vocabulary Size: {num_classes}")

In [ ]:
# 3. PyTorch Dataset
class How2SignGlossDataset(Dataset):
    def __init__(self, df, feature_dir, word_to_idx):
        self.samples = []
        for _, row in df.iterrows():
            vid_id = row['SENTENCE_NAME']
            feat_path = os.path.join(feature_dir, f"{vid_id}.npy")
            
            if os.path.exists(feat_path):
                # We need to make sure the feature isn't empty
                feat = np.load(feat_path)
                if len(feat) > 0:
                    glosses = str(row['GENERATED_GLOSS']).split()
                    target = [word_to_idx.get(w, word_to_idx["<UNK>"]) for w in glosses]
                    self.samples.append((feat, target))
                    
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        feat, target = self.samples[idx]
        return torch.tensor(feat, dtype=torch.float32), torch.tensor(target, dtype=torch.long)

def collate_fn(batch):
    # Sort by sequence length for CTC efficiency
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    sequences, targets = zip(*batch)
    
    # Sequence Lengths
    input_lengths = torch.tensor([len(s) for s in sequences], dtype=torch.long)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    
    # Pad sequences
    sequences_padded = pad_sequence(sequences, batch_first=True)
    targets_flat = torch.cat(targets)
    
    return sequences_padded, targets_flat, input_lengths, target_lengths

dataset = How2SignGlossDataset(df, FEATURE_DIR, word_to_idx)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

print(f"Total valid training samples: {len(dataset)}")

In [ ]:
# 4. Model Architecture (1D-CNN + BiLSTM)
class ASLSignToGlossModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.3):
        super(ASLSignToGlossModel, self).__init__()
        
        # Temporal Convolution (1D) to capture local motion patterns
        self.conv1d = nn.Sequential(
            nn.Conv1d(input_dim, hidden_dim, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # BiLSTM for temporal sequence modeling
        self.lstm = nn.LSTM(
            input_size=hidden_dim, 
            hidden_size=hidden_dim, 
            num_layers=2, 
            batch_first=True, 
            bidirectional=True, 
            dropout=dropout
        )
        
        # Classifier
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)
        self.log_softmax = nn.LogSoftmax(dim=-1)

    def forward(self, x):
        # x shape: (Batch, Time, Features)
        # Conv1d expects: (Batch, Features, Time)
        x = x.permute(0, 2, 1)
        x = self.conv1d(x)
        
        # LSTM expects: (Batch, Time, Features)
        x = x.permute(0, 2, 1)
        lstm_out, _ = self.lstm(x)
        
        logits = self.classifier(lstm_out)
        return self.log_softmax(logits)

# The input feature dimension is 510 (from MediaPipe extraction)
INPUT_DIM = 510
model = ASLSignToGlossModel(INPUT_DIM, HIDDEN_DIM, num_classes).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# CTC Loss MUST run on CPU because MPS does not natively support it well yet
criterion = nn.CTCLoss(blank=0, zero_infinity=True).to("cpu")

In [ ]:
# 5. Training Loop
print("Starting Training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for sequences, targets, input_lengths, target_lengths in train_loader:
        # Move data to MPS
        sequences = sequences.to(DEVICE)
        
        optimizer.zero_grad()
        
        # Forward pass (Batch, Time, NumClasses)
        outputs = model(sequences)
        
        # CTC Loss expects: (Time, Batch, NumClasses)
        outputs = outputs.permute(1, 0, 2)
        
        # Calculate loss on CPU
        loss = criterion(outputs.cpu(), targets.cpu(), input_lengths.cpu(), target_lengths.cpu())
        
        if torch.isfinite(loss):
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

print("Training Complete! Save the model weights if you are satisfied.")
# torch.save(model.state_dict(), "cv_model_weights.pth")